# 🚀 End-to-End Data Science Pipeline: Career Archetype Classification
Notebook ini dibangun untuk memenuhi kriteria silabus Data Science yang mencakup:
1. **Data Ingestion & Cleaning** (Pandas, Numpy)
2. **Exploratory Data Analysis (EDA)** (Matplotlib, Seaborn)
3. **Feature Engineering & Dimensionality Reduction** (TF-IDF Vectorization)
4. **Model Building, Training & Tuning** (Scikit-Learn: Naive Bayes / Random Forest)
5. **Model Evaluation Metrics** (Accuracy, Recall, Precision, F1-Score)
6. **Model Serialization / MLOps Deployment** (Pickle)

## 1. Import Libraries & Data Ingestion
Langkah pertama adalah mengimpor semua pustaka (libraries) wajib seperti Pandas, Matplotlib, Seaborn, dan Scikit-Learn. Kemudian kita akan membaca dataset raw.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import pickle
import warnings
warnings.filterwarnings('ignore')

# Data Ingestion (Mengambil data mentah dari CSV)
df = pd.read_csv('dataset.csv')
print(f"Total data awal: {df.shape[0]} baris")
df.head()

## 2. Data Cleaning & Wrangling
Di sini kita menangani nilai yang hilang (missing values) dan memastikan tipe data sudah benar untuk diproses oleh algoritma Machine Learning.

In [ ]:
# Menghapus baris yang memiliki nilai kosong pada kolom target atau fitur
df_cleaned = df.dropna(subset=['Category', 'Resume']).copy()

# Membersihkan teks dasar (Wrangling / Formatting)
df_cleaned['Resume'] = df_cleaned['Resume'].str.lower()

print(f"Total data setelah cleaning: {df_cleaned.shape[0]} baris")

## 3. Exploratory Data Analysis (EDA)
Menganalisis distribusi kelas (Category) untuk melihat apakah data kita *imbalanced*, dan melihat sebaran panjang karakter teks.

In [ ]:
plt.figure(figsize=(10, 6))
category_counts = df_cleaned['Category'].value_counts().head(15) # Menampilkan Top 15 Kategori
sns.barplot(x=category_counts.values, y=category_counts.index, palette='viridis')
plt.title('Top 15 Distribusi Kategori (Career Archetypes)')
plt.xlabel('Jumlah Sampel')
plt.ylabel('Kategori')
plt.show()

## 4. Feature Engineering & Splitting Data
Algoritma ML tidak bisa memproses teks secara langsung. Kita gunakan **TF-IDF Vectorizer** (Term Frequency-Inverse Document Frequency) untuk mengubah teks menjadi matriks angka. Ini juga bertindak sebagai reduksi dimensi fitur NLP.

In [ ]:
# Splitting data (80% Training, 20% Testing)
X = df_cleaned['Resume']
y = df_cleaned['Category']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Feature Engineering (TF-IDF)
# max_features membatasi dimensi agar terhindar dari curse of dimensionality
vectorizer = TfidfVectorizer(stop_words='english', max_features=5000, ngram_range=(1, 2))
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

print(f"Dimensi data latih setelah Vectorization: {X_train_vec.shape}")

## 5. Model Building & Training (Naive Bayes & Random Forest)
Kita akan melatih baseline model dan membandingkannya. Sesuai silabus, Scikit-Learn digunakan di sini.

In [ ]:
print("Training Naive Bayes Classifier...")
# Model Tuning (alpha parameter)
nb_model = MultinomialNB(alpha=0.5)
nb_model.fit(X_train_vec, y_train)

print("Training Random Forest Classifier...")
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train_vec, y_train)

print("Training Selesai!")

## 6. Koding Evaluasi Model (Metrik Klasifikasi)
Kita akan menguji kedua model menggunakan Testing Data dan mengevaluasi metriknya: **Accuracy, Precision, Recall, dan F1-Score**.

In [ ]:
# Evaluasi Naive Bayes
nb_preds = nb_model.predict(X_test_vec)
print("=== Evaluasi Naive Bayes ===")
print(f"Accuracy: {accuracy_score(y_test, nb_preds)*100:.2f}%")

# Evaluasi Random Forest
rf_preds = rf_model.predict(X_test_vec)
print("\n=== Evaluasi Random Forest ===")
print(f"Accuracy: {accuracy_score(y_test, rf_preds)*100:.2f}%")

print("\nDetailed Classification Report (Naive Bayes):")
print(classification_report(y_test, nb_preds))

## 7. Model Serialization & Deployment Preparation
Langkah terakhir dari pipeline MLOps adalah menyimpan (serialisasi) model terbaik ke bentuk file biner `.pkl` (Pickle) agar dapat di-deploy dan digunakan oleh *backend* Web Dashboard (Laravel/PHP).

In [ ]:
# Menyimpan Model dan Vectorizer
with open('tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(vectorizer, f)

with open('classifier.pkl', 'wb') as f:
    # Menyimpan Naive Bayes karena biasanya lebih ringan dan cukup akurat untuk Teks
    pickle.dump(nb_model, f)

print("Model berhasil diserialisasi dan disimpan sebagai .pkl untuk deployment MLOps!")